In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
from torchvision.io import decode_image
import os
from PIL import Image
import pandas as pd

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [16]:
zip_url="https://github.com/ishananand06/CSOT26_Computer-Vision/raw/2fd0d1ce24662f1cc9f5c0ad4d36c2163a22d973/Week%201/test_set.zip"
if os.path.exists('test_set.zip'):
    os.remove('test_set.zip')

os.makedirs('FashionMNIST/test_files', exist_ok=True)
!wget {zip_url} -O test_set.zip
!unzip -o test_set.zip -d FashionMNIST/test_files

--2026-06-05 17:49:47--  https://github.com/ishananand06/CSOT26_Computer-Vision/raw/2fd0d1ce24662f1cc9f5c0ad4d36c2163a22d973/Week%201/test_set.zip
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... --2026-06-05 17:49:47--  https://github.com/ishananand06/CSOT26_Computer-Vision/raw/2fd0d1ce24662f1cc9f5c0ad4d36c2163a22d973/Week%201/test_set.zip
Resolving github.com (github.com)... 140.82.113.3
Connecting to github.com (github.com)|140.82.113.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/ishananand06/CSOT26_Computer-Vision/2fd0d1ce24662f1cc9f5c0ad4d36c2163a22d973/Week%201/test_set.zip [following]
--2026-06-05 17:49:47--  https://raw.githubusercontent.com/ishananand06/CSOT26_Computer-Vision/2fd0d1ce24662f1cc9f5c0ad4d36c2163a22d973/Week%201/test_set.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)

In [17]:
class InferenceImageDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        self.img_names = [f for f in os.listdir(img_dir)]

    def __len__(self):
        return len(self.img_names)
    def __getitem__(self, idx):
        img_name = self.img_names[idx]
        img_path = os.path.join(self.img_dir, img_name)
        image = Image.open(img_path)
        if self.transform:
            image = self.transform(image)
        return image, img_name

In [18]:
test_transforms = v2.Compose([v2.ToImage(),
    v2.Resize((28,28)),
    v2.ToDtype(torch.float32,scale=True)
])
test_dir = "FashionMNIST/test_files/images"
test_dataset = InferenceImageDataset(img_dir=test_dir,transform=test_transforms)
test_inf_dataloader=DataLoader(test_dataset, batch_size=100, shuffle=False)

In [19]:
training_data=datasets.FashionMNIST(root="FashionMNIST",train=True,
transform=v2.Compose([v2.ToImage(),v2.RandomHorizontalFlip(p=0.5),v2.ToDtype(torch.float32,scale=True)]),download=True)

test_data=datasets.FashionMNIST(root="FashionMNIST",train=False,
transform=v2.Compose([v2.ToImage(),v2.ToDtype(torch.float32,scale=True)]),download=True)



In [20]:
batch_size=128
training_dataloader=DataLoader(training_data,batch_size=batch_size,shuffle=True)
test_dataloader=DataLoader(test_data,batch_size=batch_size,shuffle=False)

In [21]:
class conv_network(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_block_1 = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.conv_block_2 = nn.Sequential(
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.conv_block_3 = nn.Sequential(
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )


        self.flatten = nn.Flatten()
        self.classifier = nn.Sequential(
            nn.Linear(3136*2, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512,128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.conv_block_1(x)
        x = self.conv_block_2(x)
        x = self.conv_block_3(x)
        x = self.flatten(x)
        logits = self.classifier(x)
        return logits
model = conv_network().to(device)

In [ ]:
learning_rate=0.001
num_epochs=20
loss_fn=nn.CrossEntropyLoss()
optimizer=torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=5e-4, amsgrad=True)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=2)

In [23]:
def train_loop(dataloader,model,loss_fn,optimizer):
  model.train()
  size=len(dataloader.dataset)
  correct=0
  for batch,(X,y) in enumerate(dataloader):
    X, y = X.to(device), y.to(device)
    pred=model(X)
    loss=loss_fn(pred,y)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    if batch%100==0:
      loss_val=loss.item()
      current=batch*batch_size+len(X)
      print(f"loss: {loss_val:>7f}  [{current:>5d}/{size:>5d}]")
    correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    optimizer.zero_grad()
  train_accuracy = correct / size
  print(f"Training Accuracy: {(100*train_accuracy):>0.1f}%")

def test_loop(dataloader,model,loss_fn):
  model.eval()
  size = len(dataloader.dataset)
  num_batches = len(dataloader)
  test_loss, correct = 0, 0
  with torch.no_grad():
    for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred=model(X)
            test_loss+=loss_fn(pred,y).item()
            correct+=(pred.argmax(1)==y).type(torch.float).sum().item()
  test_loss/=num_batches
  correct/=size
  print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
  return test_loss

In [11]:
for t in range(num_epochs):
    print(f"Epoch {t+1}")
    train_loop(training_dataloader,model,loss_fn,optimizer)
    vloss=test_loop(test_dataloader,model,loss_fn)
    scheduler.step(vloss)


Epoch 1
loss: 2.373923  [  128/60000]
loss: 0.569327  [12928/60000]
loss: 0.507975  [25728/60000]
loss: 0.396868  [38528/60000]
loss: 0.293266  [51328/60000]
Training Accuracy: 83.7%
Test Error: 
 Accuracy: 87.6%, Avg loss: 0.334007 

Epoch 2
loss: 0.359781  [  128/60000]
loss: 0.404737  [12928/60000]
loss: 0.350207  [25728/60000]
loss: 0.294724  [38528/60000]
loss: 0.224122  [51328/60000]
Training Accuracy: 89.3%
Test Error: 
 Accuracy: 90.9%, Avg loss: 0.256543 

Epoch 3
loss: 0.178259  [  128/60000]
loss: 0.318800  [12928/60000]
loss: 0.316412  [25728/60000]
loss: 0.224953  [38528/60000]
loss: 0.260370  [51328/60000]
Training Accuracy: 90.8%
Test Error: 
 Accuracy: 90.3%, Avg loss: 0.262776 

Epoch 4
loss: 0.350873  [  128/60000]
loss: 0.230730  [12928/60000]
loss: 0.259697  [25728/60000]
loss: 0.288381  [38528/60000]
loss: 0.167818  [51328/60000]
Training Accuracy: 91.6%
Test Error: 
 Accuracy: 91.6%, Avg loss: 0.224982 

Epoch 5
loss: 0.228254  [  128/60000]
loss: 0.172984  [12928

In [12]:

model.eval()
image_ids=[]
predictions=[]
with torch.no_grad():
    for images, img_names in test_inf_dataloader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, dim=1)
        image_ids.extend(img_names)
        predictions.extend(preds.cpu().numpy())
print(f"Finished predicting")

Finished predicting
Finished predicting


In [13]:
results=pd.DataFrame({
    'image_id':image_ids,
    'label':predictions
})
results=results.sort_values(by='image_id').reset_index(drop=True)
results.to_csv('2025MT11352_CNN.csv', index=False)
print("predictions successfully saved")

predictions successfully saved
predictions successfully saved
